In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from datetime import datetime
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch.nn.functional as F
import gc

In [14]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS = 300
LR = 1e-5

In [15]:
class MELDFusionDataset(Dataset):
    def __init__(self, label_encoder, csv_path, audio_path, video_path, text_path):
        self.df = pd.read_csv(csv_path)
        self.label_encoder = label_encoder
        self.audio_path = audio_path
        self.video_path = video_path
        self.text_path = text_path
        self.text_embeddings_cache = None
        self._load_text_embeddings()

        self.emotion_map = {
            'neutral': 0,
            'clam': 1,
            'happy': 2,
            'sad': 3,
            'angry': 4,
            'fearful': 5,
            'disgusted': 6,
            'surprised': 7
        }

    def _load_text_embeddings(self):
        """Load text embeddings once at initialization"""
        text_embeddings_path = os.path.join(self.text_path, "text_embeddings.pt")
        
        if os.path.exists(text_embeddings_path):
            self.text_embeddings_cache = torch.load(text_embeddings_path)
        else:
            self.text_embeddings_cache = {}
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = self.emotion_map[row["emotion"]]
        file_name = row["file_name"]
        audio_input_path = os.path.join(self.audio_path, f"{file_name}.npy")
        video_input_path = os.path.join(self.video_path, f"{file_name}.npy")

        if os.path.exists(audio_input_path):
            audio_input = torch.from_numpy(np.load(audio_input_path)).float()
        else:
            audio_input = torch.zeros(768)

        if os.path.exists(video_input_path):
            video_input = torch.from_numpy(np.load(video_input_path)).float()
        else:
            video_input = torch.zeros(2048)

        text_input = torch.zeros(384)
        
        if self.text_embeddings_cache:
            stat_id = str(row['stat_id'])
            
            if stat_id in self.text_embeddings_cache:
                text_input = self.text_embeddings_cache[stat_id].float()
            else:
                print(f"⚠️  Statement ID {stat_id} not found in text embeddings")

        return {
            "text_input" : text_input,
            "video_input" : video_input,
            "audio_input" : audio_input,
            "labels" : torch.tensor(label, dtype = torch.long)
        }

In [16]:
class FusionClassifier(nn.Module):

    def __init__(
        self,
        text_dim=384,
        audio_dim=768,
        video_dim=2048,
        hidden_dim=128,
        num_of_classes=8
    ):
        super().__init__()

        # -------- Modality Encoders -------- #

        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.video_encoder = nn.Sequential(
            nn.Linear(video_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # -------- Attention Fusion -------- #

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 3)  # one weight per modality
        )

        # -------- Classifier -------- #

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_of_classes)
        )

    def forward(self, text_emb, audio_emb, video_emb):

        t = self.text_encoder(text_emb)
        a = self.audio_encoder(audio_emb)
        v = self.video_encoder(video_emb)

        # Concatenate for attention weight calculation
        concat = torch.cat([t, a, v], dim=1)

        weights = torch.softmax(
            self.attention(concat),
            dim=1
        )

        # Split weights
        wt = weights[:, 0:1]
        wa = weights[:, 1:2]
        wv = weights[:, 2:3]

        # Weighted fusion
        fused = wt * t + wa * a + wv * v

        return self.classifier(fused)


In [17]:
def evaluate(model, dataloader, epoch, device, criterion):
    """Evaluate model on validation set"""
    model.eval()  # Set to eval mode
    y_true, y_pred = [], []
    total_loss = 0
    
    with torch.no_grad():  # No gradients during evaluation
        for batch in dataloader:
            text_emb = batch["text_input"].to(device)
            audio_emb = batch["audio_input"].to(device)
            video_emb = batch["video_input"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(text_emb, audio_emb, video_emb)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.cpu().numpy())
    
    loss = total_loss / len(dataloader)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    
    return loss, acc, f1

In [18]:
text_path = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Train/embeddings/Text/text_embeddings.pt"
csv_train = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Train/ravdess_train.csv"
audio_train = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Train/embeddings/Audio"
video_train = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Train/embeddings/Video"
csv_dev = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Train/ravdess_dev.csv"
csv_test = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Test/ravdess_mulitmodal_Test.csv"
audio_test = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Test/embeddings/Audio"
video_test = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/data/Test/embeddings/Video"


In [19]:
le = LabelEncoder()
train_df = pd.read_csv(csv_train)
le.fit(train_df["emotion"])

train_set = MELDFusionDataset(le,csv_train, audio_train, video_train, text_path)
dev_set = MELDFusionDataset(le, csv_dev, audio_train, video_train, text_path)
test_set = MELDFusionDataset(le, csv_test, audio_test, video_test, text_path)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
dev_loader = DataLoader(dev_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
model = FusionClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing= 0.1)

# Setup directories
LOG_DIR = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/logs"
SAVE_DIR = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

log_file = os.path.join(LOG_DIR, "training_log.txt")
checkpoint_path = os.path.join(SAVE_DIR, "fusion_checkpoint.pt")
best_model_path = os.path.join(SAVE_DIR, "best_model.pt")

# Initialize log file
with open(log_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("Training Started\n")
    f.write(f"Device: {DEVICE}\n")
    f.write(f"Learning Rate: {LR}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write("="*80 + "\n\n")

def log_metrics(epoch, train_loss, train_acc, train_f1, dev_loss, dev_acc, dev_f1, is_best=False):
    """Log metrics to file and console"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    message = (
        f"[{timestamp}] Epoch {epoch+1:3d}/{EPOCHS} | "
        f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f}, F1={train_f1:.4f} | "
        f"Dev Loss={dev_loss:.4f}, Acc={dev_acc:.4f}, F1={dev_f1:.4f} | "
    )
    
    if is_best:
        message += "BEST"
    
    message += "\n"
    
    # Write to file
    with open(log_file, 'a') as f:
        f.write(message)
    
    # Print to console
    print(message.strip())

# Load checkpoint if exists
start_epoch = 0
best_dev_f1 = 0

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_dev_f1 = checkpoint.get("best_dev_f1", 0)
    print(f"✅ Resumed training from epoch {start_epoch}")
    print(f"   Best F1 so far: {best_dev_f1:.4f}")

print(f"\nStarting training from epoch {start_epoch + 1}\n")

# Training loop
for epoch in range(start_epoch, EPOCHS):
    # Training phase
    model.train()
    y_true, y_pred = [], []
    total_loss = 0
    batch_count = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
        optimizer.zero_grad()
        
        # Move to device
        text_emb = batch["text_input"].to(DEVICE)
        audio_emb = batch["audio_input"].to(DEVICE)
        video_emb = batch["video_input"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        # Forward pass
        logits = model(text_emb, audio_emb, video_emb)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        total_loss += loss.item()
        batch_count += 1
        
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.cpu().numpy())
    
    # Calculate training metrics
    train_loss = total_loss / batch_count
    train_acc = accuracy_score(y_true, y_pred)
    train_f1 = f1_score(y_true, y_pred, average="weighted")
    
    # Validation phase
    model.eval()
    dev_loss, dev_acc, dev_f1 = evaluate(model, dev_loader, epoch, DEVICE, criterion)
    
    # Check if best model
    is_best = dev_f1 > best_dev_f1
    if is_best:
        best_dev_f1 = dev_f1
        # Save best model
        torch.save(model.state_dict(), best_model_path)
    
    # Save checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_dev_f1": best_dev_f1
    }, checkpoint_path)
    
    # Log metrics
    log_metrics(epoch, train_loss, train_acc, train_f1, dev_loss, dev_acc, dev_f1, is_best=is_best)
    
    # Clear GPU cache every epoch
    torch.cuda.empty_cache()
    gc.collect()

# Training completed
print("\n" + "="*80)
print("Training Completed!")
print(f"Best F1 Score: {best_dev_f1:.4f}")
print(f"Checkpoint saved: {checkpoint_path}")
print(f"Best model saved: {best_model_path}")
print(f"Log file: {log_file}")
print("="*80)

# Append final summary to log
with open(log_file, 'a') as f:
    f.write("\n" + "="*80 + "\n")
    f.write("Training Completed!\n")
    f.write(f"Best F1 Score: {best_dev_f1:.4f}\n")
    f.write("="*80 + "\n")


✅ Resumed training from epoch 200
   Best F1 so far: 0.6536

Starting training from epoch 201



[2026-02-13 22:11:33] Epoch 201/300 | Train Loss=1.3007, Acc=0.6302, F1=0.6210 | Dev Loss=1.2798, Acc=0.6653, F1=0.6520


[2026-02-13 22:11:38] Epoch 202/300 | Train Loss=1.3091, Acc=0.6274, F1=0.6194 | Dev Loss=1.2796, Acc=0.6653, F1=0.6537BEST


[2026-02-13 22:11:42] Epoch 203/300 | Train Loss=1.2972, Acc=0.6354, F1=0.6287 | Dev Loss=1.2778, Acc=0.6694, F1=0.6565BEST


[2026-02-13 22:11:47] Epoch 204/300 | Train Loss=1.3035, Acc=0.6281, F1=0.6195 | Dev Loss=1.2727, Acc=0.6722, F1=0.6608BEST


...

[2026-02-13 22:18:49] Epoch 296/300 | Train Loss=1.1538, Acc=0.7281, F1=0.7257 | Dev Loss=1.1364, Acc=0.7250, F1=0.7211


[2026-02-13 22:18:53] Epoch 297/300 | Train Loss=1.1538, Acc=0.7076, F1=0.7056 | Dev Loss=1.1383, Acc=0.7250, F1=0.7206


[2026-02-13 22:18:58] Epoch 298/300 | Train Loss=1.1540, Acc=0.7108, F1=0.7080 | Dev Loss=1.1363, Acc=0.7319, F1=0.7285


[2026-02-13 22:19:03] Epoch 299/300 | Train Loss=1.1588, Acc=0.7170, F1=0.7154 | Dev Loss=1.1387, Acc=0.7167, F1=0.7128


[2026-02-13 22:19:08] Epoch 300/300 | Train Loss=1.1534, Acc=0.7160, F1=0.7138 | Dev Loss=1.1399, Acc=0.7139, F1=0.7071

Training Completed!
Best F1 Score: 0.7304
Checkpoint saved: C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/checkpoints\fusion_checkpoint.pt
Best model saved: C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/checkpoints\best_model.pt
Log file: C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/logs\training_log.txt


In [21]:
model_path = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/ravdess_model/model/checkpoints/best_model.pt"
checkpoint = torch.load(model_path, map_location= DEVICE)
model = FusionClassifier()
model.load_state_dict(checkpoint)

<All keys matched successfully>

In [ ]:
def evaluate(model, dataloader, device, criterion):

    model.eval()

    y_true, y_pred = [], []
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(dataloader):

            text_emb  = batch["text_input"].to(device)
            audio_emb = batch["audio_input"].to(device)
            video_emb = batch["video_input"].to(device)
            labels    = batch["labels"].to(device)

            logits = model(text_emb, audio_emb, video_emb)

            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.cpu().numpy())

    loss = total_loss / total_samples

    acc = accuracy_score(y_true, y_pred)
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    f1_macro = f1_score(y_true, y_pred, average="macro")

    return loss, acc, f1_weighted, f1_macro


In [23]:
criterion = nn.CrossEntropyLoss()
loss, acc, f1, f1_micro = evaluate(model=model, dataloader = test_loader, device=DEVICE, criterion=criterion)
print(f"loss: {loss}, acc: {acc}, f1: {f1}, f1_micro: {f1_micro}")

100%|██████████| 90/90 [00:00<00:00, 148.46it/s]

loss: 1.825988214338819, acc: 0.3763888888888889, f1: 0.35062899262111097, f1_micro: 0.3435173121612389


In [24]:
# print("Testing Logistic Regression...\n")

# # Prepare training data
# X_train = []
# y_train = []

# for batch in train_loader:
#     text = batch["text_input"].cpu().numpy()
#     audio = batch["audio_input"].cpu().numpy()
#     video = batch["video_input"].cpu().numpy()
    
#     # Concatenate: (384 + 768 + 2048 = 3200 features)
#     x = np.concatenate([text, audio, video], axis=1)
#     X_train.append(x)
#     y_train.append(batch["labels"].cpu().numpy())

# X_train = np.vstack(X_train)
# y_train = np.concatenate(y_train)

# print(f"Training data shape: {X_train.shape}")

# # Train logistic regression
# lr_model = LogisticRegression(
#     max_iter=5000,
#     random_state=42,
#     class_weight='balanced',
#     n_jobs=-1,  # Use all cores
#     verbose=1
# )
# lr_model.fit(X_train, y_train)

# # Prepare dev data
# X_dev = []
# y_dev = []

# for batch in dev_loader:
#     text = batch["text_input"].cpu().numpy()
#     audio = batch["audio_input"].cpu().numpy()
#     video = batch["video_input"].cpu().numpy()
    
#     x = np.concatenate([text, audio, video], axis=1)
#     X_dev.append(x)
#     y_dev.append(batch["labels"].cpu().numpy())

# X_dev = np.vstack(X_dev)
# y_dev = np.concatenate(y_dev)

# print(f"Dev data shape: {X_dev.shape}\n")

# # Evaluate
# y_pred = lr_model.predict(X_dev)

# dev_f1 = f1_score(y_dev, y_pred, average='weighted')
# dev_acc = accuracy_score(y_dev, y_pred)

# print(f"Logistic Regression Results:")
# print(f"  Dev F1: {dev_f1:.4f}")
# print(f"  Dev Acc: {dev_acc:.4f}")

# # Prepare test data
# X_test = []
# y_test = []

# for batch in test_loader:
#     text = batch["text_input"].cpu().numpy()
#     audio = batch["audio_input"].cpu().numpy()
#     video = batch["video_input"].cpu().numpy()
    
#     x = np.concatenate([text, audio, video], axis=1)
#     X_test.append(x)
#     y_test.append(batch["labels"].cpu().numpy())

# X_test = np.vstack(X_test)
# y_test = np.concatenate(y_test)

# # Test predictions
# y_test_pred = lr_model.predict(X_test)

# test_f1 = f1_score(y_test, y_test_pred, average='weighted')
# test_acc = accuracy_score(y_test, y_test_pred)

# print(f"\nLogistic Regression Test Results:")
# print(f"  Test F1: {test_f1:.4f}")
# print(f"  Test Acc: {test_acc:.4f}")

# print(f"\nClassification Report:")
# print(classification_report(y_test, y_test_pred, 
#                           target_names=['neutral', 'calm', 'happy', 'sad', 
#                                        'angry', 'fearful', 'disgusted', 'surprised']))

# # Save model
# import pickle
# with open('logistic_regression_model.pkl', 'wb') as f:
#     pickle.dump(lr_model, f)
# print("\nModel saved!")